In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

staff = pd.read_excel(file_name)

staff["employee_id"]       = pd.to_numeric(staff["employee_id"], errors="coerce").astype("Int64")
staff["weekly_hours"]      = pd.to_numeric(staff["weekly_hours"], errors="coerce")
staff["avg_hourly_rate"]   = pd.to_numeric(staff["avg_hourly_rate"], errors="coerce").fillna(50)
staff["is_dreammaker"]     = staff["is_dreammaker"].astype(bool)
staff["is_kantoor"]        = staff["is_kantoor"].astype(bool)
staff["eligible_projects"] = staff["eligible_projects"].apply(
    lambda v: [x.strip() for x in str(v).split(",") if x.strip()] if pd.notna(v) else [])
staff["available_days"]    = staff["available_days"].apply(
    lambda v: [x.strip() for x in str(v).split(",") if x.strip()] if pd.notna(v)
    else ["Monday","Tuesday","Wednesday","Thursday","Friday"])

print(f"Total employees: {len(staff)}")

Saving staff_template.xlsx to staff_template.xlsx
Total employees: 71


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df_proj = pd.read_excel(file_name)
df_proj = df_proj[[c for c in df_proj.columns if not str(c).startswith("Unnamed")]]
df_proj["hours_per_week"] = pd.to_numeric(df_proj["hours_per_week"], errors="coerce")
df_proj["headcount"]      = pd.to_numeric(df_proj["headcount"],      errors="coerce").fillna(1)

type_avg = df_proj.groupby("funding_type")["hours_per_week"].mean()
def fill_hours(row):
    if pd.notna(row["hours_per_week"]): return row["hours_per_week"]
    return round(type_avg.get(row["funding_type"], np.nan), 1)
df_proj["hours_per_week"] = df_proj.apply(fill_hours, axis=1)

def parse_list(val):
    if pd.isna(val) or str(val).strip() == "": return []
    return [v.strip() for v in str(val).split(",") if v.strip()]

def parse_time(val):
    if pd.isna(val): return None
    try: return datetime.strptime(str(val)[:5], "%H:%M")
    except: return None

def calc_session_hours(start, end):
    try:
        s = datetime.strptime(str(start)[:5], "%H:%M")
        e = datetime.strptime(str(end)[:5],   "%H:%M")
        return (e - s).seconds / 3600
    except: return None

df_proj["days_list"]         = df_proj["days"].apply(parse_list)
df_proj["start_parsed"]      = df_proj["start_time"].apply(parse_time)
df_proj["end_parsed"]        = df_proj["end_time"].apply(parse_time)
df_proj["hours_per_session"] = df_proj.apply(
    lambda r: calc_session_hours(r["start_time"], r["end_time"]), axis=1)
df_proj["row_demand"] = df_proj["hours_per_week"] * df_proj["headcount"]

D              = df_proj.groupby("project")["row_demand"].sum().to_dict()
proj_headcount = df_proj.groupby("project")["headcount"].max().to_dict()

project_schedule = {}
for _, row in df_proj.iterrows():
    p = row["project"]
    if p not in project_schedule: project_schedule[p] = []
    for day in row["days_list"]:
        h = row["hours_per_session"] or (row["hours_per_week"] / max(len(row["days_list"]),1))
        project_schedule[p].append({
            "day"      : day,
            "headcount": int(row["headcount"]),
            "start"    : row["start_parsed"],
            "end"      : row["end_parsed"],
            "start_str": str(row["start_time"])[:5] if pd.notna(row["start_time"]) else "",
            "end_str"  : str(row["end_time"])[:5]   if pd.notna(row["end_time"])   else "",
            "hours"    : float(h) if h and not np.isnan(float(h)) else 0.0
        })

project_days = {p: list(set(s["day"] for s in sch))
                for p, sch in project_schedule.items()}

print(f"Total projects        : {df_proj['project'].nunique()}")
print(f"Total weekly demand   : {sum(D.values()):.1f}h")

Saving project_requirements_template.xlsx to project_requirements_template.xlsx
Total projects        : 26
Total weekly demand   : 557.5h


In [ ]:
import pulp
import pandas as pd
import numpy as np
from datetime import datetime

PLANNING_WEEK   = "2025-W1"
FREELANCER_RATE = 200  # High penalty to minimize freelancer usage
DAYS = ["Monday","Tuesday","Wednesday","Thursday","Friday"]

# ── Helper functions ──────────────────────────────────────────
def get_project_type(project):
    if project.startswith("Combibrug CC"):    return "CC"
    elif project.startswith("Combibrug BSC"): return "BSC"
    elif project.startswith("MDT"):           return "MDT"
    elif "MP" in project:                     return "Combiworld-MP"
    elif project.startswith("Combiworld"):    return "Combiworld"
    return ""

def times_conflict(s1, e1, s2, e2, buffer=1.0):
    if not all([s1, e1, s2, e2]): return False
    if s1 > s2: s1, e1, s2, e2 = s2, e2, s1, e1
    gap = (s2 - e1).seconds / 3600
    return gap < buffer

# ── Parameters ───────────────────────────────────────────────
E         = staff["employee_id"].tolist()
P         = list(D.keys())
A         = dict(zip(staff["employee_id"], staff["weekly_hours"].fillna(0)))
max_daily = {e: A.get(e, 0) / 5 for e in E}
c         = dict(zip(staff["employee_id"], staff["avg_hourly_rate"].fillna(50)))

role_map          = dict(zip(staff["employee_id"], staff["role"].fillna("").str.lower()))
is_kantoor_map    = dict(zip(staff["employee_id"], staff["is_kantoor"]))
is_dreammaker_map = dict(zip(staff["employee_id"], staff["is_dreammaker"]))
avail_days_map    = dict(zip(staff["employee_id"], staff["available_days"]))

def get_emp_avail_days(emp_id):
    v = avail_days_map.get(emp_id, DAYS)
    if isinstance(v, str): return [d.strip() for d in v.split(",")]
    return v if isinstance(v, list) else DAYS

def get_elig_list(emp_id):
    v = staff.loc[staff["employee_id"]==emp_id, "eligible_projects"].values
    if len(v)==0: return []
    return v[0] if isinstance(v[0], list) else [x.strip().upper() for x in str(v[0]).split(",") if x.strip()]

def is_eligible(emp_id, project):
    if is_kantoor_map.get(emp_id, False): return 0
    emp_elig = get_elig_list(emp_id)
    ft = get_project_type(project).upper().replace("-","")
    return 1 if any(ft.startswith(e.replace("-","")) or
                    e.replace("-","").startswith(ft) for e in emp_elig) else 0

def has_day_overlap(emp_id, project):
    emp_days  = get_emp_avail_days(emp_id)
    proj_days = project_days.get(project, [])
    if not proj_days: return True
    return bool(set(emp_days) & set(proj_days))

def get_hours_if_assigned(emp_id, project):
    emp_days = get_emp_avail_days(emp_id)
    slots    = project_schedule.get(project, [])
    if not slots:
        d  = D.get(project, 0)
        hc = proj_headcount.get(project, 1)
        val = d / max(1, hc) if (d and hc and not np.isnan(float(d))) else 0.0
        return float(val)
    total = 0.0
    for slot in slots:
        if slot["day"] in emp_days:
            h = slot.get("hours", 0)
            if h and not np.isnan(float(h)):
                total += float(h)
    return total

elig = {(e,p): is_eligible(e,p) * has_day_overlap(e,p) for e in E for p in P}

dreammakers     = [e for e in E if is_dreammaker_map.get(e, False)]
PL_KW           = ["locatie leider","proj.l"]
project_leaders = [e for e in E if any(k in str(role_map.get(e,"")) for k in PL_KW)]
mdt_cw_projects = [p for p in P if get_project_type(p) in ("MDT","Combiworld","Combiworld-MP")]
bsc_projects    = [p for p in P if get_project_type(p) == "BSC"]

# C6: conflict pairs (gap < 1 hour = conflict)
conflicts_set = set()
for day in DAYS:
    day_slots = [(p, sl["start"], sl["end"])
                 for p, sch in project_schedule.items()
                 for sl in sch if sl["day"] == day]
    for i in range(len(day_slots)):
        for j in range(i+1, len(day_slots)):
            p1,s1,e1 = day_slots[i]; p2,s2,e2 = day_slots[j]
            if p1 != p2 and times_conflict(s1,e1,s2,e2):
                conflicts_set.add((day, min(p1,p2), max(p1,p2)))

print(f"Planning week   : {PLANNING_WEEK}")
print(f"Employees       : {len(E)}")
print(f"Projects        : {len(P)}")
print(f"Dreammakers     : {dreammakers}")
print(f"Project leaders : {project_leaders}")
print(f"Eligible (e,p)  : {sum(elig.values()):,} / {len(E)*len(P):,}")
print(f"Time conflicts  : {len(conflicts_set)}")

# ── Build model ───────────────────────────────────────────────
model = pulp.LpProblem("Combibrug_Workforce_Planning", pulp.LpMinimize)

y = {(e,p): pulp.LpVariable(f"y_{e}_{P.index(p)}", cat="Binary")
     for e in E for p in P if elig[(e,p)]==1 and D.get(p,0)>0}

s = {p: pulp.LpVariable(f"s_{P.index(p)}", lowBound=0)
     for p in P if D.get(p,0)>0}

print(f"y: {len(y):,}  s: {len(s):,}")

# Objective
model += (
    pulp.lpSum(c.get(e,0) * get_hours_if_assigned(e,p) * y[(e,p)] for (e,p) in y) +
    pulp.lpSum(FREELANCER_RATE * s[p] for p in s)
)

# C1: availability
for e in E:
    terms = [get_hours_if_assigned(e,p) * y[(e,p)] for p in P if (e,p) in y]
    if terms: model += pulp.lpSum(terms) <= A.get(e,0), f"C1_e{e}"

# C2: demand
for p in P:
    d = D.get(p,0)
    if d > 0:
        terms_p = [get_hours_if_assigned(e,p) * y[(e,p)] for e in E if (e,p) in y]
        if terms_p:
            model += pulp.lpSum(terms_p) + s.get(p,0) >= d, f"C2_p{P.index(p)}"

# C3: max daily hours
for e in E:
    mdh = max_daily.get(e, 8)
    for p in P:
        if (e,p) in y:
            emp_days = get_emp_avail_days(e)
            for slot in [sl for sl in project_schedule.get(p,[]) if sl["day"] in emp_days]:
                if slot.get("hours",0) > mdh:
                    model += y[(e,p)] == 0, f"C3_e{e}_p{P.index(p)}"
                    break

# C4: four-eyes for MDT/Combiworld
for p in mdt_cw_projects:
    if D.get(p,0) > 0:
        ay = [y[(e,p)] for e in E if (e,p) in y]
        if ay: model += pulp.lpSum(ay) >= 2, f"C4a_p{P.index(p)}"
        dy = [y[(e,p)] for e in dreammakers if (e,p) in y]
        if dy: model += pulp.lpSum(dy) >= 1, f"C4b_p{P.index(p)}"

# C5: project leader for BSC
for p in bsc_projects:
    if D.get(p,0) > 0:
        pl_v = [y[(e,p)] for e in project_leaders if (e,p) in y]
        if pl_v: model += pulp.lpSum(pl_v) >= 1, f"C5_p{P.index(p)}"

# C6: no conflicting assignments (gap < 1 hour)
for day, p1, p2 in conflicts_set:
    for e in E:
        if (e,p1) in y and (e,p2) in y:
            model += y[(e,p1)] + y[(e,p2)] <= 1, f"C6_e{e}_d{day}_p{P.index(p1)}vp{P.index(p2)}"

print(f"Constraints: {len(model.constraints):,}")

# ── Solve ─────────────────────────────────────────────────────
status = model.solve(pulp.PULP_CBC_CMD(msg=True, timeLimit=300))
print(f"\nStatus    : {pulp.LpStatus[model.status]}")
print(f"Total cost: €{pulp.value(model.objective):,.0f}")

# ── Extract results ───────────────────────────────────────────
records = []
for (e,p), var in y.items():
    val = pulp.value(var)
    if val and val > 0.5:
        emp_row = staff[staff["employee_id"]==e]
        hours   = get_hours_if_assigned(e, p)
        records.append({
            "employee_id"            : e,
            "role"                   : emp_row["role"].values[0] if len(emp_row)>0 else None,
            "worker_type"            : emp_row["worker_type"].values[0] if len(emp_row)>0 else None,
            "project"                : p,
            "funding"                : get_project_type(p),
            "hours"                  : round(hours, 1),
            "cost_eur"               : round(hours * c.get(e,0), 2),
            "contract_hours_per_week": emp_row["weekly_hours"].values[0] if len(emp_row)>0 else None
        })

results = pd.DataFrame(records)

fl_records = []
for p, var in s.items():
    val = pulp.value(var)
    if val and val > 0.01:
        fl_records.append({
            "project"                  : p,
            "funding"                  : get_project_type(p),
            "freelancer_hours"         : round(val,1),
            "freelancer_cost_estimated": round(val * FREELANCER_RATE, 2)
        })
freelancer_df = pd.DataFrame(fl_records)

# ── Output Table 1: Daily schedule ───────────────────────────
schedule_records = []
for (e,p), var in y.items():
    val = pulp.value(var)
    if val and val > 0.5:
        slots    = project_schedule.get(p,[])
        emp_days = get_emp_avail_days(e)
        if slots:
            for slot in slots:
                if slot["day"] in emp_days:
                    schedule_records.append({
                        "day"        : slot["day"],
                        "start_time" : slot.get("start_str",""),
                        "end_time"   : slot.get("end_str",""),
                        "project"    : p,
                        "employee_id": e,
                        "hours"      : round(slot.get("hours",0),1)
                    })
        else:
            schedule_records.append({
                "day": "—", "start_time": "", "end_time": "",
                "project": p, "employee_id": e,
                "hours": round(get_hours_if_assigned(e,p),1)
            })

day_order   = {d:i for i,d in enumerate(DAYS)}
schedule_df = pd.DataFrame(schedule_records)
if not schedule_df.empty:
    schedule_df["day_order"] = schedule_df["day"].map(day_order)
    schedule_df = schedule_df.sort_values(["day_order","start_time","project","employee_id"])

print("\n=== Table 1: Daily schedule ===")
if not schedule_df.empty:
    print(schedule_df[["day","start_time","end_time","project","employee_id","hours"]].to_string(index=False))
else:
    print("No schedule found.")

print("\n=== Table 2: Employee summary ===")
if not results.empty:
    emp_summary = results.groupby("employee_id").agg(
        role                    = ("role", "first"),
        worker_type             = ("worker_type", "first"),
        projects                = ("project", lambda x: ", ".join(sorted(x.unique()))),
        total_hours             = ("hours", "sum"),
        contract_hours_per_week = ("contract_hours_per_week", "first"),
        total_cost_eur          = ("cost_eur", "sum")
    ).reset_index()
    print(emp_summary.to_string(index=False))

print("\n=== Table 3: Freelancer requirements ===")
if not freelancer_df.empty:
    print(freelancer_df[["project","freelancer_hours","freelancer_cost_estimated"]]
          .sort_values("project").to_string(index=False))
else:
    print("No freelancers needed.")

internal_cost   = results["cost_eur"].sum()                        if not results.empty      else 0
freelancer_cost = freelancer_df["freelancer_cost_estimated"].sum() if not freelancer_df.empty else 0

print("\n=== Summary ===")
print(f"Planning week              : {PLANNING_WEEK}")
print(f"Internal cost              : €{internal_cost:,.0f}")
print(f"Freelancer cost (estimate) : €{freelancer_cost:,.0f}")
print(f"Total cost                 : €{internal_cost + freelancer_cost:,.0f}")
print(f"Projects fully staffed     : {sum(1 for p in P if D.get(p,0)>0 and (pulp.value(s.get(p,0)) or 0)<0.01)} / {sum(1 for p in P if D.get(p,0)>0)}")
print(f"Projects needing freelancer: {len(freelancer_df)}")

Planning week   : 2025-W1
Employees       : 71
Projects        : 26
Dreammakers     : [36, 38, 42, 45, 46, 47, 48, 51, 58, 72]
Project leaders : [53, 60, 61]
Eligible (e,p)  : 700 / 1,846
Time conflicts  : 36
y: 700  s: 26
Constraints: 795

Status    : Optimal
Total cost: €19,939

=== Table 1: Daily schedule ===
      day start_time end_time                                 project  employee_id  hours
   Monday      08:00    13:30  Combibrug CC - Daltonschool Oegstgeest           23    5.5
   Monday      08:00    13:30  Combibrug CC - Daltonschool Oegstgeest           27    5.5
   Monday      08:00    13:30  Combibrug CC - Daltonschool Oegstgeest           30    5.5
   Monday      08:00    13:30                   Combibrug CC - De Ley           23    5.5
   Monday      08:00    13:30                   Combibrug CC - De Ley           61    5.5
   Monday      08:00    12:00                Combiworld - Capelle AMV           19    4.0
   Monday      08:00    12:00                Combiworld 